In [1]:
import numpy as np
from pynq import Overlay, allocate
import time
import inspect

class CNN_Tester:

    ADDR_IMAGE_WIDTH    = 0x00
    ADDR_INPUT_CHANNEL  = 0x04
    ADDR_OUTPUT_CHANNEL = 0x08
    ADDR_STRIDE         = 0x0C
    ADDR_PADDING        = 0x10
    ADDR_START          = 0x14
    ADDR_EOC_STATUS     = 0x18

    def __init__(self, bitstream_file):
        
        # PYNQ 오버레이 로드
        self.bitstream_file = bitstream_file
        print(f"[1/2] PYNQ 오버레이 로딩 ({bitstream_file})...")
        
        # 기본 세팅
        try:
            self.overlay = Overlay(bitstream_file)
            self.conv_ip = self.overlay.Conv_IP_0 # 본 연구에서 설계한 IP 이름
            self.dma_image = self.overlay.axi_dma_0 # dma
            self.dma_filter = self.overlay.axi_dma_1
            self.dma_output = self.overlay.axi_dma_2
            print("SUCCESS : PYNQ 환경 설정 완료")
        except Exception as e:
            print(f"ERROR : PYNQ 설정 중 오류 발생: {e}")
            raise

        # 데이터 타입 정의
        self.DTYPE_IN = np.int8
        self.DTYPE_IN_BIAS = np.int32
        self.DTYPE_OUT = np.int32
        
        # 테스트 파라미터 저장을 위한 변수
        self.params = {}

    def _generate_data(self):
        # 설정된 파라미터에 맞춰 랜덤 정수 테스트 데이터를 생성
        p = self.params
        
        IMAGE_MIN, IMAGE_MAX = -128, 127 # 입력 값 범위 설정
        FILTER_MIN, FILTER_MAX = -128, 127 # 필터 값 범위 설정
        BIAS_MIN, BIAS_MAX = -1000000000, 1000000000 # 바이어스 값 범위 설정

        image_shape = (p['input_channel'], p['image_width'], p['image_width'])
        weights_shape = (p['input_channel'], p['output_channel'], p['kernel_size'], p['kernel_size'])
        biases_shape = (p['output_channel'],)

        image_np = np.random.randint(IMAGE_MIN, IMAGE_MAX + 1, size=image_shape, dtype=self.DTYPE_IN)
        weights_np = np.random.randint(FILTER_MIN, FILTER_MAX + 1, size=weights_shape, dtype=self.DTYPE_IN)
        biases_np = np.random.randint(BIAS_MIN, BIAS_MAX + 1, size=biases_shape, dtype=self.DTYPE_IN_BIAS)
        
        weights_flat = weights_np.flatten()
        filter_data_np = np.concatenate((weights_flat, biases_np)).astype(self.DTYPE_IN_BIAS)
        
        return image_np, weights_np, biases_np, filter_data_np

    def _run_golden_model(self, image, weights, biases):
        # for-loop 사용해 컨볼루션의 Golden 값을 계산
        print("\n[B] Golden Model (For-loop) 실행...")
        
        p = self.params
        C_in, H_in, W_in = image.shape
        _, F_out, K, _ = weights.shape
        S, P = p['stride'], p['padding']

        H_out = (H_in - K + 2 * P) // S + 1
        W_out = (W_in - K + 2 * P) // S + 1

        image_padded = np.pad(image, ((0, 0), (P, P), (P, P)), 'constant')
        output = np.zeros((H_out, W_out, F_out), dtype=self.DTYPE_OUT)

        image_padded_l = image_padded.astype(np.int64)
        weights_l = weights.astype(np.int64)
        biases_l = biases.astype(np.int64)
        
        start_time = time.time()
        
        for oc in range(F_out):
            for oh in range(H_out):
                for ow in range(W_out):
                    accumulator = 0
                    for ic in range(C_in):
                        for kh in range(K):
                            for kw in range(K):
                                ih = oh * S + kh
                                iw = ow * S + kw
                                weight_val = weights_l[ic, oc, kh, kw]
                                image_val = image_padded_l[ic, ih, iw]
                                accumulator += image_val * weight_val
                    
                    accumulator += biases_l[oc]
                    output[oh, ow, oc] = accumulator
        
        duration_ms = (time.time() - start_time) * 1000
        
        golden_result = output.flatten()
    
        print(f"  - Golden 결과 생성 완료 (소요 시간: {duration_ms:.4f} ms)")
        print(f"  - 예상 출력 크기: {golden_result.shape}")
        return golden_result

    def _run_fpga(self, image_np, filter_data_np, output_shape):
        # FPGA 가속기를 사용하여 컨볼루션 연산을 수행
        print("\n[C] FPGA 가속기 실행...")
        
        image_buffer = None
        filter_buffer = None
        output_buffer = None
        
        try:
            # PYNQ DMA 버퍼 할당
            image_buffer = allocate(shape=image_np.shape, dtype=self.DTYPE_IN)
            filter_buffer = allocate(shape=filter_data_np.shape, dtype=self.DTYPE_IN_BIAS)
            output_buffer = allocate(shape=output_shape, dtype=self.DTYPE_OUT)
            
            np.copyto(image_buffer, image_np)
            np.copyto(filter_buffer, filter_data_np)
            output_buffer.fill(0)

            # AXI-Lite를 통해 IP에 파라미터 설정
            p = self.params
            self.conv_ip.write(self.ADDR_IMAGE_WIDTH, p['image_width'])
            self.conv_ip.write(self.ADDR_INPUT_CHANNEL, p['input_channel'])
            self.conv_ip.write(self.ADDR_OUTPUT_CHANNEL, p['output_channel'])
            self.conv_ip.write(self.ADDR_STRIDE, p['stride'])
            self.conv_ip.write(self.ADDR_PADDING, p['padding'])
            
            start_time = time.time()
            
            # DMA 전송 시작
            self.dma_output.recvchannel.transfer(output_buffer)
            self.dma_filter.sendchannel.transfer(filter_buffer)
            self.dma_image.sendchannel.transfer(image_buffer)
            
            # 연산 실행 트리거
            self.conv_ip.write(self.ADDR_START, 1)
            
            # EOC 신호 대기
            while not (self.conv_ip.read(self.ADDR_EOC_STATUS) & 0x1):
                pass

            # 모든 DMA 전송 대기
            self.dma_filter.sendchannel.wait()
            self.dma_image.sendchannel.wait()
            self.dma_output.recvchannel.wait()

            duration_ms = (time.time() - start_time) * 1000
            print(f"  - FPGA 연산 완료 (소요 시간: {duration_ms:.4f} ms)")
            
            return output_buffer

        except RuntimeError as e:
            print(f"ERROR : FPGA 실행 중 런타임 오류 발생: {e}")
            if "Allocate failed" in str(e):
                print("    -> 메모리 할당 실패")
            frame = inspect.currentframe()
            if frame:
                print(f"    -> 오류 발생 지점: line {frame.f_lineno}")
            return None
            
        finally:
            # 할당된 버퍼가 있다면 메모리 해제
            if image_buffer is not None: 
                image_buffer.freebuffer()
            if filter_buffer is not None: 
                filter_buffer.freebuffer()
            # output_buffer는 반환되므로 여기서 해제 X

    def _verify_results(self, golden, fpga):
        # Golden과 FPGA 결과를 비교
        print("\n[D] 결과 검증...")
        
        if fpga is None:
            print("실패 : FPGA 실행 중 오류 발생")
            return

        if np.array_equal(golden, fpga):
            print("성공: FPGA 결과가 Golden과 완벽히 일치합니다!")
            print("\n--- 결과 샘플 (앞 10개) ---")
            print(f"{'Index':<10}{'Golden':<15}{'FPGA':<15}")
            print("-" * 40)
            for i in range(min(10, len(golden))):
                print(f"{i:<10}{golden[i]:<15}{fpga[i]:<15}")
        else:
            print("실패: FPGA 결과가 Golden과 다릅니다.")
            diff_indices = np.where(golden != fpga)[0]
            print(f"  - 총 {len(golden)}개 중 {len(diff_indices)}개의 불일치 발견.")
            print("\n--- 불일치 샘플 (최대 10개) ---")
            print(f"{'Index':<10}{'Golden':<15}{'FPGA':<15}")
            print("-" * 40)
            for i, idx in enumerate(diff_indices):
                if i >= 10: break
                print(f"{idx:<10}{golden[idx]:<15}{fpga[idx]:<15}")

    def run_test(self, test_params):
        # 주어진 파라미터로 테스트(데이터 생성, 골든 계산, FPGA, 비교)를 실행
        
        print("=" * 60)
        print("새로운 테스트 실행 시작")
        print("=" * 60)
        
        # 1. 파라미터 설정
        self.params = test_params
        print("[A] 테스트 파라미터 설정:")
        for key, value in self.params.items():
            print(f"  - {key:<15}: {value}")
        
        fpga_result = None
        
        try:
            # 2. 데이터 생성
            print("\n[A] 테스트 데이터 생성 중...")
            image, weights, biases, filter_data = self._generate_data()
            print("  - 데이터 생성 완료.")
            
            # 3. 골든 실행
            golden_result = self._run_golden_model(image, weights, biases)
            
            # 4. FPGA 실행
            fpga_result = self._run_fpga(image, filter_data, golden_result.shape)
            
            # 5. 결과 검증
            self._verify_results(golden_result, fpga_result)
            
        except Exception as e:
            print(f"\n테스트 실행 중 예기치 않은 오류가 발생했습니다: {e}")
            import traceback
            traceback.print_exc()
        finally:
            # FPGA 실행이 성공했다면(None이 아니라면) output_buffer의 메모리를 해제
            if fpga_result is not None:
                fpga_result.freebuffer()
                
            print("\n" + "=" * 60)
            print("테스트 실행 완료")
            print("=" * 60 + "\n")

print("CNN_Tester 클래스가 성공적으로 정의되었습니다.")

CNN_Tester 클래스가 성공적으로 정의되었습니다.


In [2]:
BITSTREAM_FILE = 'cnn.bit'
# 비트스트림을 로드하고 테스터 객체를 생성

try:
    tester = CNN_Tester(BITSTREAM_FILE)
except Exception as e:
    print("테스터 객체 생성에 실패했습니다. PYNQ 보드 연결 및 파일 경로 확인")

[1/2] PYNQ 오버레이 로딩 (cnn.bit)...
SUCCESS : PYNQ 환경 설정 완료


In [3]:
# 테스트 파라미터 수정
test_config = {
    'image_width'   : 32,
    'input_channel' : 16,
    'output_channel': 32,
    'kernel_size'   : 3, #fix
    'stride'        : 5,
    'padding'       : 2,
}


# 설정된 파라미터로 테스트 실행
if 'tester' in locals():
    tester.run_test(test_config)
else:
    print("오류: 이전 셀을 먼저 실행하여 'tester' 객체를 생성해야 합니다.")

새로운 테스트 실행 시작
[A] 테스트 파라미터 설정:
  - image_width    : 32
  - input_channel  : 16
  - output_channel : 32
  - kernel_size    : 3
  - stride         : 5
  - padding        : 2

[A] 테스트 데이터 생성 중...
  - 데이터 생성 완료.

[B] Golden Model (For-loop) 실행...
  - Golden 결과 생성 완료 (소요 시간: 2127.7223 ms)
  - 예상 출력 크기: (1568,)

[C] FPGA 가속기 실행...
  - FPGA 연산 완료 (소요 시간: 2.4726 ms)

[D] 결과 검증...
성공: FPGA 결과가 Golden과 완벽히 일치합니다!

--- 결과 샘플 (앞 10개) ---
Index     Golden         FPGA           
----------------------------------------
0         -652656550     -652656550     
1         292337905      292337905      
2         -540760567     -540760567     
3         162221706      162221706      
4         -139159021     -139159021     
5         352863540      352863540      
6         173123678      173123678      
7         -31881940      -31881940      
8         -273926901     -273926901     
9         -925513409     -925513409     

테스트 실행 완료

